# TurboQuant Visualized
**Paper**: [TurboQuant: Online Vector Quantization with Near-optimal Distortion Rate](https://arxiv.org/abs/2504.19874)

The story in 6 cells:
1. The rotation trick
2. Scalar quantization (1D k-means)
3. The bias problem (MSE ≠ good for inner products)
4. The two-stage fix
5. Near-optimality (vs Shannon lower bound)
6. Why it matters (KV cache)

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.cluster.vq import kmeans
np.random.seed(42)
%matplotlib inline
plt.rcParams['figure.figsize'] = (15, 5)

## 1. The Rotation Trick
Randomly rotate any vector on the unit sphere → each coordinate becomes ~Gaussian and nearly independent.
This lets us quantize each coordinate separately.

In [ ]:
# --- Math: Random rotation ---
d = 500
x = np.zeros(d); x[0] = 1.0  # worst case: all mass in one dim

# Random rotation via QR decomposition
Pi, _ = np.linalg.qr(np.random.randn(d, d))
y = Pi @ x  # rotated

# Check independence across many vectors
samples = np.random.randn(2000, d)
samples /= np.linalg.norm(samples, axis=1, keepdims=True)
rot = (Pi @ samples.T).T

print(f'Before rotation: x[0]={x[0]}, x[1]={x[1]}')
print(f'After rotation:  y[0]={y[0]:.4f}, y[1]={y[1]:.4f}')
print(f'Std of rotated coords: {y.std():.4f} (expected {1/np.sqrt(d):.4f})')
print(f'Correlation(coord 0, coord 1): {np.corrcoef(rot[:,0], rot[:,1])[0,1]:.4f} (≈ 0)')

In [ ]:
# --- Plot: Rotation trick ---
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].bar(range(20), x[:20], color='#e74c3c')
axes[0].set_title('Before rotation (worst-case)'); axes[0].set_ylim(-0.15, 1.1)
axes[1].bar(range(20), y[:20], color='#3498db')
axes[1].set_title('After rotation (first 20 coords)'); axes[1].set_ylim(-0.15, 0.15)
axes[2].hist(y, bins=40, density=True, color='#3498db', alpha=0.7, label='Rotated coords')
xx = np.linspace(-0.15, 0.15, 200)
axes[2].plot(xx, norm.pdf(xx, 0, 1/np.sqrt(d)), 'r-', lw=2, label=f'N(0, 1/{d})')
axes[2].set_title('Distribution → Gaussian'); axes[2].legend()
plt.tight_layout(); plt.show()

## 2. Scalar Quantization
Each coordinate is ~Gaussian. Quantize by partitioning into 2^b buckets and snapping to nearest centroid.
This is just 1D k-means (Lloyd-Max) on the Gaussian.

In [ ]:
# --- Math: Scalar quantization (1D k-means) ---
sigma = 1 / np.sqrt(d)
quantizers = {}  # store for plotting

for b in [1, 2, 3, 4]:
    samp = np.random.randn(100000) * sigma
    centroids, _ = kmeans(samp, 2**b)
    centroids = np.sort(centroids)
    q = centroids[np.argmin(np.abs(samp[:, None] - centroids[None, :]), axis=1)]
    mse = np.mean((samp - q)**2)
    quantizers[b] = {'centroids': centroids, 'mse': mse}
    print(f'b={b}: {2**b} centroids = {centroids.round(4)}, MSE={mse:.5f}')

In [ ]:
# --- Plot: Scalar quantization ---
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for idx, b in enumerate([1, 2, 3, 4]):
    ax = axes[idx]
    xx = np.linspace(-0.15, 0.15, 300)
    ax.fill_between(xx, norm.pdf(xx, 0, sigma), alpha=0.2, color='gray')
    colors = plt.cm.Set2(np.linspace(0, 1, 2**b))
    for i, c_val in enumerate(quantizers[b]['centroids']):
        ax.axvline(c_val, color=colors[i], lw=2, ls='--')
    ax.set_title(f'b={b} ({2**b} centroids)\nMSE={quantizers[b]["mse"]:.5f}')
plt.suptitle('More bits → more centroids → less error', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

## 3. The Bias Problem
MSE-optimal quantizers produce **biased** inner product estimates.
At 1 bit, inner products are overestimated by factor 2/π ≈ 0.64. Attention scores in transformers would be wrong.

In [ ]:
# --- Math: The bias problem ---
n = 3000
rand_unit = lambda n, d: (v := np.random.randn(n, d)) / np.linalg.norm(v, axis=1, keepdims=True)
xs, ys = rand_unit(n, d), rand_unit(n, d)
true_ip = np.sum(xs * ys, axis=1)

# 1-bit MSE quantize: rotate → sign → scale by centroid → rotate back
xs_rot = (Pi @ xs.T).T
c = np.sqrt(2 / (np.pi * d))
xs_deq = (Pi.T @ (np.sign(xs_rot) * c).T).T
est_ip = np.sum(ys * xs_deq, axis=1)
err = est_ip - true_ip
lims = [-0.15, 0.15]

slope = np.polyfit(true_ip, est_ip, 1)[0]
print(f'Bias slope: {slope:.3f} (expected 2/π = {2/np.pi:.3f})')
print(f'Mean error: {err.mean():.4f} (should be 0 if unbiased)')

In [ ]:
# --- Plot: Bias problem ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(true_ip, est_ip, alpha=0.1, s=3, color='#e74c3c')
axes[0].plot(lims, lims, 'k--', lw=1.5, label='Perfect (y=x)')
axes[0].plot(lims, [l*2/np.pi for l in lims], 'b-', lw=2, label=f'Bias (slope=2/π≈{2/np.pi:.2f})')
axes[0].set(xlim=lims, ylim=lims, xlabel='True ⟨x,y⟩', ylabel='Estimated ⟨x,y⟩',
            title='1-bit MSE Quantizer: Systematic bias!')
axes[0].legend(); axes[0].set_aspect('equal')
axes[1].hist(err, bins=50, density=True, color='#e74c3c', alpha=0.7)
axes[1].axvline(0, color='k', ls='--'); axes[1].axvline(err.mean(), color='red', lw=2)
axes[1].set(xlabel='Error (est - true)', title=f'Mean error = {err.mean():.4f} (should be 0)')
plt.tight_layout(); plt.show()

## 4. The Two-Stage Fix
**Stage 1**: MSE quantize with (b-1) bits  
**Stage 2**: QJL (random sign projection) on the residual with 1 bit  
Same total bit budget → **unbiased** inner product estimator!

In [ ]:
# --- Math: Two-stage fix ---
# Stage 1: 1-bit MSE (same as above)
xs_mse = (Pi.T @ (np.sign(xs_rot) * c).T).T
est_mse_only = np.sum(ys * xs_mse, axis=1)

# Stage 2: QJL on residual
residuals = xs - xs_mse
r_norms = np.linalg.norm(residuals, axis=1)
S = np.random.randn(d, d)
qjl_signs = np.sign((S @ residuals.T).T)
xs_qjl = np.sqrt(np.pi/2) / d * (S.T @ qjl_signs.T).T * r_norms[:, None]

# Combined estimate
est_combined = np.sum(ys * (xs_mse + xs_qjl), axis=1)
err_mse = est_mse_only - true_ip
err_fix = est_combined - true_ip

print(f'MSE-only mean error:  {err_mse.mean():.4f} (biased)')
print(f'Two-stage mean error: {err_fix.mean():.4f} (unbiased!)')

In [ ]:
# --- Plot: Two-stage fix ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(true_ip, est_mse_only, alpha=0.1, s=3, color='#e74c3c', label='MSE only (biased)')
axes[0].scatter(true_ip, est_combined, alpha=0.1, s=3, color='#2ecc71', label='Two-stage (unbiased)')
axes[0].plot(lims, lims, 'k--', lw=2)
axes[0].set(xlim=lims, ylim=lims, xlabel='True ⟨x,y⟩', ylabel='Estimated',
            title='Same 2-bit budget: MSE-only vs Two-stage')
axes[0].legend(); axes[0].set_aspect('equal')
axes[1].hist(err_mse, bins=50, density=True, alpha=0.5, color='#e74c3c', label=f'MSE only (mean={err_mse.mean():.4f})')
axes[1].hist(err_fix, bins=50, density=True, alpha=0.5, color='#2ecc71', label=f'Two-stage (mean={err_fix.mean():.4f})')
axes[1].axvline(0, color='k', ls='--')
axes[1].set(xlabel='Error', title='Two-stage is centered on 0!'); axes[1].legend()
plt.tight_layout(); plt.show()

## 5. Near-Optimality
TurboQuant is within ~2.7× of the information-theoretic lower bound (Shannon).  
No algorithm — no matter how complex — can do better than the lower bound.

In [ ]:
# --- Math: Near-optimality ---
bits = np.array([1, 2, 3, 4, 5])
lower = 1 / 4**bits                                      # Shannon lower bound
turbo = np.array([0.36, 0.117, 0.03, 0.009, 0.009/4])   # TurboQuant (from paper)

for b, l, t in zip(bits, lower, turbo):
    print(f'b={b}: lower bound={l:.5f}, TurboQuant={t:.5f}, gap={t/l:.1f}×')

In [ ]:
# --- Plot: Near-optimality ---
fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(bits, lower, 's--', color='#2ecc71', lw=2, ms=10, label='Lower bound (impossible to beat)')
ax.semilogy(bits, turbo, 'o-', color='#3498db', lw=2, ms=10, label='TurboQuant')
ax.fill_between(bits, lower, turbo, alpha=0.15, color='#3498db')
for i, b in enumerate(bits[:4]):
    ax.annotate(f'{turbo[i]/lower[i]:.1f}×', xy=(b, np.sqrt(turbo[i]*lower[i])),
                fontsize=12, ha='center', color='#e74c3c', fontweight='bold')
ax.set(xlabel='Bit-width', ylabel='MSE Distortion (log)', title='Gap to theoretical limit')
ax.legend(fontsize=12); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 6. Why It Matters: KV Cache Memory
In transformer inference, the KV cache grows linearly with context length.  
At 100K tokens, it's ~12.5 GB. TurboQuant brings it to ~2.7 GB with **zero quality loss**.

In [ ]:
# --- Math: KV cache memory ---
ctx = np.array([1, 4, 16, 32, 64, 100, 128]) * 1000
# Llama-3.1-8B: 32 layers, 8 KV heads, d_head=128, fp16
mem_full = ctx * 32 * 8 * 128 * 2 * 2 / 1e9  # GB

# Paper results (Table 1, LongBench-E avg score)
methods = ['Full (16b)', 'KIVI (3b)', 'Polar (3.9b)', 'Turbo (2.5b)', 'Turbo (3.5b)']
scores = [50.06, 48.50, 49.78, 49.44, 50.06]

print(f'KV cache at 100K tokens: {mem_full[5]:.1f} GB (full) → {mem_full[5]*3.5/16:.1f} GB (3.5-bit)')
for m, s in zip(methods, scores):
    print(f'  {m}: score={s}')

In [ ]:
# --- Plot: KV cache ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(ctx/1000, mem_full, 'o-', color='#e74c3c', lw=2, label='16-bit (full)')
axes[0].plot(ctx/1000, mem_full * 3.5/16, 's-', color='#3498db', lw=2, label='3.5-bit (TurboQuant)')
axes[0].plot(ctx/1000, mem_full * 2.5/16, '^-', color='#2ecc71', lw=2, label='2.5-bit (TurboQuant)')
axes[0].axhline(80, color='gray', ls=':', lw=2); axes[0].text(5, 83, 'A100 (80GB)', color='gray')
axes[0].axhline(24, color='gray', ls=':', lw=1.5); axes[0].text(5, 26, 'RTX 4090 (24GB)', color='gray')
axes[0].set(xlabel='Context (K tokens)', ylabel='KV Cache (GB)', title='Memory grows linearly')
axes[0].legend()
labels = ['Full\n(16b)', 'KIVI\n(3b)', 'Polar\n(3.9b)', 'Turbo\n(2.5b)', 'Turbo\n(3.5b)']
clrs = ['#95a5a6', '#e74c3c', '#f39c12', '#3498db', '#3498db']
axes[1].bar(labels, scores, color=clrs, alpha=0.8, edgecolor='black')
axes[1].set(ylabel='LongBench Avg Score', title='Quality: TurboQuant 3.5b = Full precision')
axes[1].set_ylim(47, 51); axes[1].axhline(50.06, color='gray', ls='--', alpha=0.5)
plt.tight_layout(); plt.show()